# Chapter 5: Multiple Regression Analysis — OLS Asymptotics

Topics: large-sample properties of OLS, LM test, simulation with non-normal errors.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import wooldridge as woo
import matplotlib.pyplot as plt
import scipy.stats as stats
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 5.1 Simulation: Effect of Sample Size on β̂ Distribution

In [ ]:
np.random.seed(42)
true_b1 = 0.5
sample_sizes = [25, 100, 500]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, n in zip(axes, sample_sizes):
    betas = []
    for _ in range(2000):
        x = np.random.uniform(0, 10, n)
        u = np.random.normal(0, 2, n)
        y = 1 + true_b1 * x + u
        m = smf.ols('y ~ x', data=pd.DataFrame({'y':y,'x':x})).fit()
        betas.append(m.params['x'])
    ax.hist(betas, bins=50, edgecolor='black', color='steelblue', density=True)
    ax.axvline(true_b1, color='red', lw=2)
    ax.set_title(f'n = {n}\nstd(β̂) = {np.std(betas):.4f}')
    ax.set_xlabel('β̂₁')

plt.suptitle('Sampling Distribution of β̂₁ by Sample Size', fontsize=13)
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_asymptotics.png", dpi=150)
plt.show()

## 5.1.2 Non-Normal Error Terms

In [ ]:
np.random.seed(42)
n, n_sims = 200, 2000

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, error_type in zip(axes, ['Normal', 'Chi-squared(1)']):
    betas = []
    for _ in range(n_sims):
        x = np.random.uniform(0, 10, n)
        u = np.random.normal(0, 2, n) if error_type == 'Normal' else (np.random.chisquare(1, n) - 1)
        y = 1 + 0.5 * x + u
        m = smf.ols('y ~ x', data=pd.DataFrame({'y':y,'x':x})).fit()
        betas.append(m.params['x'])
    ax.hist(betas, bins=50, edgecolor='black', color='steelblue', density=True)
    ax.axvline(0.5, color='red', lw=2, label='True β₁')
    ax.set_title(f'Error: {error_type}')
    ax.legend()

plt.tight_layout()
plt.savefig(f"{output_dir}/fig2_non_normal_errors.png", dpi=150)
plt.show()

## 5.2 LM Test (Lagrange Multiplier / RESET)

In [ ]:
df = woo.dataWoo('wage1')
model = smf.ols('wage ~ educ + exper + tenure', data=df).fit()

from statsmodels.stats.diagnostic import linear_reset
reset_test = linear_reset(model, power=2, use_f=True)
print(f"RESET test F-stat: {reset_test.statistic:.4f}")
print(f"RESET test p-value: {reset_test.pvalue:.4f}")
print("\nIf p < 0.05: evidence of functional form misspecification")